In [0]:
# Ingesting single CSV file to bronze table

from pyspark.sql.functions import col, current_timestamp

path = "/Volumes/ecommerce/bronze/raw_files/olist_orders_dataset.csv"
orders_df = (spark.read
             .option("header", True)
             .option("inferSchema", True)
             .option("quote", '"')
             .csv(path)
             .withColumn("_source_file", col("_metadata.file_path"))
             .withColumn("_ingested_at", current_timestamp())
             )
display(orders_df)

In [0]:
# Looping through list of CSV files to bronze tables

VOLUME = "/Volumes/ecommerce/bronze/raw_files"

files = [
    ("olist_customers_dataset.csv",            "customers"),
    ("olist_geolocation_dataset.csv",          "geolocation"),
    ("olist_order_items_dataset.csv",          "order_items"),
    ("olist_order_payments_dataset.csv",       "order_payments"),
    ("olist_order_reviews_dataset.csv",        "order_reviews"),
    ("olist_orders_dataset.csv",               "orders"),
    ("olist_products_dataset.csv",             "products"),
    ("olist_sellers_dataset.csv",              "sellers"),
    ("product_category_name_translation.csv",  "product_category_translations"),
]

for filename, table in files:
    df = (spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .option("quote", '"')
        .option("escape", '"')
        .option("multiLine", "true")   # reviews file has newlines in comment text
        .option("encoding", "UTF-8")   # Portuguese characters in reviews
        .csv(f"{VOLUME}/{filename}")
        .withColumn("_source_file", col("_metadata.file_path"))
        .withColumn("_ingested_at", col("_metadata.file_modification_time"))
    )

    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"ecommerce.bronze.{table}")
    table_count = spark.read.table(f"ecommerce.bronze.{table}").count()
    print(f"{table}: {df.count()}/{table_count} rows")

In [0]:
spark.read.table("ecommerce.bronze.orders").count() # Confirm table and csv count match
spark.read.table("ecommerce.bronze.order_reviews").select("_source_file", "_ingested_at").show(3) # Confirm metadata columns are present
spark.read.table("ecommerce.bronze.order_reviews").select("review_comment_message").show(25, truncate=False) # Confirm UTF-8 characters are present